In [8]:
import os 
from typing import List, Dict, Any
import pandas as pd 


In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
print("Set up completed")

Set up completed


## Understanding Document Structure In Langchain

In [ ]:
## create a simple document 
doc = Document(
    page_content="This is the main test content that will be embadded and searched.",
    metadata = {
        "source":"example.txt",
        "page": 1,
        "author": "Rahul Sharma",
        "date_created" : "2026-03-07",
        "custom_field":"any_value"
    }
)
print("Document Structure")

print(f"Content : {doc.page_content}")

print(f"MetaData : {doc.metadata}")

Document Structure
Content : This is the main test content that will be embadded and searched.
Content : {'source': 'example.txt', 'page': 1, 'author': 'Rahul Sharma', 'date_created': '2026-03-07', 'custom_field': 'any_value'}


In [18]:
type(doc)

langchain_core.documents.base.Document

## Text Files - The simplest Case


In [22]:
## create a simple txt file 
import os
os.makedirs("data/text_files", exist_ok=True)
print("Folder ready: data/text_files")

Folder ready: data/text_files


In [23]:
sample_texts ={
    "data/text_files/Data.txt" : """Introduction to Python
Python is a popular, beginner-friendly programming language used in web development,
data science, machine learning, automation, and more.
Why Python is great for beginners:
1. Easy-to-read syntax
2. Large community and learning resources
3. Huge ecosystem of libraries (NumPy, pandas, scikit-learn, etc.)
Basic concepts in Python:
- Variables and data types (int, float, str, bool)
- Control flow (if/else, loops)
- Functions and modules
- Lists, dictionaries, tuples, and sets
- File handling and exception handling
Example:
name = "Rahul"
print(f"Hello, {name}! Welcome to Python.")
Python is a great first step for building AI and RAG applications.
"""
}

for filepath,content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

### Text loader - Read single file 

In [27]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("data/text_files/Data.txt", encoding="utf-8")

documents = loader.load()
print(type(documents))
print(documents)
print(f" Loaded {len(documents)} documents")
print(f"Content Preview : {documents[0].page_content[:100]}...")
print(f"MetaData : {documents[0].metadata}")

<class 'list'>
[Document(metadata={'source': 'data/text_files/Data.txt'}, page_content='Introduction to Python\nPython is a popular, beginner-friendly programming language used in web development,\ndata science, machine learning, automation, and more.\nWhy Python is great for beginners:\n1. Easy-to-read syntax\n2. Large community and learning resources\n3. Huge ecosystem of libraries (NumPy, pandas, scikit-learn, etc.)\nBasic concepts in Python:\n- Variables and data types (int, float, str, bool)\n- Control flow (if/else, loops)\n- Functions and modules\n- Lists, dictionaries, tuples, and sets\n- File handling and exception handling\nExample:\nname = "Rahul"\nprint(f"Hello, {name}! Welcome to Python.")\nPython is a great first step for building AI and RAG applications.\n')]
 Loaded 1 documents
Content Preview : Introduction to Python
Python is a popular, beginner-friendly programming language used in web devel...
MetaData : {'source': 'data/text_files/Data.txt'}


### Directory Loader - Multiple Files


In [33]:
from langchain_community.document_loaders import DirectoryLoader


## load all the text files from directory 

dir_loader = DirectoryLoader(
    "data/text_files",
    glob ="**/*.txt", ## pattern match
    loader_cls= TextLoader, ## loader class to use 
    loader_kwargs={'encoding':'utf-8'},
    show_progress= True
)

documents = dir_loader.load()

print(f" Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\n Document {i+1}")
    print(f" Source: {doc.metadata["source"]}")
    print(f" Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 1118.18it/s]

 Loaded 2 documents

 Document 1
 Source: data\text_files\Data.txt
 Length: 676 characters

 Document 2
 Source: data\text_files\Random_Text.txt
 Length: 305 characters


## Documents Splitter  / Text Splitter


In [39]:
### Different text spilliting Strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\Data.txt'}, page_content='Introduction to Python\nPython is a popular, beginner-friendly programming language used in web development,\ndata science, machine learning, automation, and more.\nWhy Python is great for beginners:\n1. Easy-to-read syntax\n2. Large community and learning resources\n3. Huge ecosystem of libraries (NumPy, pandas, scikit-learn, etc.)\nBasic concepts in Python:\n- Variables and data types (int, float, str, bool)\n- Control flow (if/else, loops)\n- Functions and modules\n- Lists, dictionaries, tuples, and sets\n- File handling and exception handling\nExample:\nname = "Rahul"\nprint(f"Hello, {name}! Welcome to Python.")\nPython is a great first step for building AI and RAG applications.\n'), Document(metadata={'source': 'data\\text_files\\Random_Text.txt'}, page_content='The quiet street slowly filled with the sound of morning traffic as the sun rose above the buildings. People walked hurriedly with coffee cups in h

In [38]:
## Character Text Spillter
text = documents[0].page_content
text

'Introduction to Python\nPython is a popular, beginner-friendly programming language used in web development,\ndata science, machine learning, automation, and more.\nWhy Python is great for beginners:\n1. Easy-to-read syntax\n2. Large community and learning resources\n3. Huge ecosystem of libraries (NumPy, pandas, scikit-learn, etc.)\nBasic concepts in Python:\n- Variables and data types (int, float, str, bool)\n- Control flow (if/else, loops)\n- Functions and modules\n- Lists, dictionaries, tuples, and sets\n- File handling and exception handling\nExample:\nname = "Rahul"\nprint(f"Hello, {name}! Welcome to Python.")\nPython is a great first step for building AI and RAG applications.\n'

In [42]:
# Method 1 : Character based splitting

print("1 CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n", # split on new line
    chunk_size=120, # Max Chunk size
    chunk_overlap=20, # Overlap bw chunk 
    length_function=len, # How to measure chunk size 
)

char_chunks = char_splitter.split_text(text)

print(f" Created {len(char_chunks)} chunk")
print(f"First Chunk : {char_chunks[0][:100]}...")



1 CHARACTER TEXT SPLITTER
 Created 7 chunk
First Chunk : Introduction to Python
Python is a popular, beginner-friendly programming language used in web devel...


In [44]:
print(char_chunks[0])
print("--------------------")
print(char_chunks[1])
print("----------------------")
print(char_chunks[2])

Introduction to Python
Python is a popular, beginner-friendly programming language used in web development,
--------------------
data science, machine learning, automation, and more.
Why Python is great for beginners:
1. Easy-to-read syntax
----------------------
2. Large community and learning resources
3. Huge ecosystem of libraries (NumPy, pandas, scikit-learn, etc.)


## Recursive Character text splitter

In [46]:
print("\n2) RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First Chunk : {recursive_chunks[0][:100]}...")



2) RECURSIVE CHARACTER TEXT SPLITTER
Created 7 chunks
First Chunk : Introduction to Python
Python is a popular, beginner-friendly programming language used in web devel...


In [47]:
print(recursive_chunks[0])
print("----------------")
print(recursive_chunks[1])

Introduction to Python
Python is a popular, beginner-friendly programming language used in web development,
----------------
data science, machine learning, automation, and more.
Why Python is great for beginners:
1. Easy-to-read syntax


In [48]:
# Different text splitting strategies (correct version)
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
sample_text = """
Python is a popular programming language.
It is widely used for web development, data science, machine learning, and automation.
LangChain helps build LLM-powered applications by combining loaders, splitters, embeddings, and vector stores.
Text splitting is important because LLMs have context limits, and smaller chunks improve retrieval quality.
"""
print("1) CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
)
char_chunks = char_splitter.split_text(sample_text)
print(f"Character chunks: {len(char_chunks)}")
for i, c in enumerate(char_chunks, 1):
    print(f"\n[Char Chunk {i}]\n{c}")
print("\n2) RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    length_function=len,
)
recursive_chunks = recursive_splitter.split_text(sample_text)
print(f"Recursive chunks: {len(recursive_chunks)}")
for i, c in enumerate(recursive_chunks, 1):
    print(f"\n[Recursive Chunk {i}]\n{c}")
print("\n3) TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=40,
    chunk_overlap=5,
)
token_chunks = token_splitter.split_text(sample_text)
print(f"Token chunks: {len(token_chunks)}")
for i, c in enumerate(token_chunks, 1):
    print(f"\n[Token Chunk {i}]\n{c}")

1) CHARACTER TEXT SPLITTER
Character chunks: 4

[Char Chunk 1]
Python is a popular programming language.

[Char Chunk 2]
It is widely used for web development, data science, machine learning, and automation.

[Char Chunk 3]
LangChain helps build LLM-powered applications by combining loaders, splitters, embeddings, and vector stores.

[Char Chunk 4]
Text splitting is important because LLMs have context limits, and smaller chunks improve retrieval quality.

2) RECURSIVE CHARACTER TEXT SPLITTER
Recursive chunks: 4

[Recursive Chunk 1]
Python is a popular programming language.

[Recursive Chunk 2]
It is widely used for web development, data science, machine learning, and automation.

[Recursive Chunk 3]
LangChain helps build LLM-powered applications by combining loaders, splitters, embeddings, and vector stores.

[Recursive Chunk 4]
Text splitting is important because LLMs have context limits, and smaller chunks improve retrieval quality.

3) TOKEN TEXT SPLITTER
Token chunks: 2

[Token Chu